In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import utils_multi as um
n_351 = um.critical_density()

plot_dir = "/ccc/cont002/home/dam/barlodun/2601a_figures_for_paper"
filename = "multi_output.bin"
input_filename = "multi_input.txt"
root = "/ccc/scratch/cont002/dam/barlodun/2504c_multi/Data"

aspect_ratio = (4, 3)

In [ ]:
list_labels = [
               r"Gopalaswarmy et al. 2024", #$\Omega$,
               "T0, normal incidence rays",
               r"T0, SG=5.0, Rb/Rt=0.76, $f_{CBET}$=0.33", #OMEGA like configuration #Rb/Rt=0.76
               r"T0, SG=6.0, Rb/Rt=0.25, $f_{CBET}$=0.33", #zooming 120Mbars #M48, SG=6.0, Rb/Rt=0.25 (zoom)
               r"T0, SG=6.0, Rb/Rt=0.25, $f_{CBET}$=0.17", #zooming 150Mbars
               r"T0, SG=5.0, Rb/Rt=0.76, $f_{CBET}$=0", # ($\delta \lambda$=0.5%) +zooming
               r"T0, SG=2.5, Rb/Rt=0.45, $f_{CBET}$=0", # ($\delta \lambda$=1.6%)
               r"T0, SG=6.0, Rb/Rt=0.25, $f_{CBET}$=0", # ($\delta \lambda$=1.6%) +zooming
              ]
list_dirs = [
             "Data_2501b_select_from_omega_benchmark/260126f_gopalaswamy_2023_naturephysics_laser_08_and_flux_01_beamspot_0067_lower_gas_density_big_picket_new_picket_change_eos",
             "Data_2504b_lafon_new_pulse/250411a_lafon_recreate_pulse",
             "Data_2504b_lafon_new_pulse/250525b_lafon_recreate_pulse_beam_1757um_lower_intensity",
             "Data_2504b_lafon_new_pulse/250624b_lafon_recreate_pulse_beam_700um_between_lower_intensity", #"Data_2504b_lafon_new_pulse/250526b_lafon_recreate_pulse_beam_700um_lower_intensity",
             "Data_2504b_lafon_new_pulse/250628a_lafon_recreate_pulse_beam_700um_500TW",
             "Data_2504b_lafon_new_pulse/250411c_lafon_recreate_pulse_beam_1757um",
             "Data_2504b_lafon_new_pulse/250628b_lafon_recreate_pulse_beam_1046um",
             "Data_2504b_lafon_new_pulse/250411e_lafon_recreate_pulse_beam_700um",
             """
             "Data_2504b_lafon_new_pulse/250628b_lafon_recreate_pulse_beam_1046um_power_drop",
             "Data_2504b_lafon_new_pulse/250628c_lafon_recreate_pulse_beam_700um_power_drop",
             """
             ]

ndirs = len(list_labels)
multi_data = {}

for idir in range(ndirs):
    label = list_labels[idir]
    loc_dir = list_dirs[idir]
    multi_data[label] = {}
    multi_data[label] = um.multi_read_bin(root+"/"+loc_dir+"/"+filename, multi_data[label])
    multi_data[label] = um.read_inputs(root+"/"+loc_dir+"/"+input_filename, multi_data[label])
    multi_data[label]["n_crit"] = um.critical_density(wavelength_l=multi_data[label]["wavelength"])
    labels = list(multi_data[label].keys())
    print("Number time outputs: ", np.shape(multi_data[label]["TIME"])[0])
    print("Number radial outputs: ", np.shape(multi_data[label]["CMC"])[1])
    print("Number fuel cells: ", multi_data[label]["fuel_boundary"])
    multi_data[label] = um.multi_data_units(multi_data[label])
    multi_data[label] = um.multi_find_interfaces(multi_data[label])
    multi_data[label] = um.multi_critical_surface(multi_data[label], multi_data[label]["n_crit"])
    multi_data[label] = um.multi_mean_laser_dep_radius(multi_data[label])
    multi_data[label] = um.define_capsule(multi_data[label])
    multi_data[label] = um.define_implosion_velocity(multi_data[label])
    multi_data[label] = um.adiabat(multi_data[label])
    multi_data[label] = um.calc_rhor(multi_data[label])
    multi_data[label] = um.lawson_criteria(multi_data[label])
    print("Printout info for file: " + loc_dir)
    um.multi_printout(multi_data[label])

In [ ]:
list_colours = [
                "tab:orange",
                "tab:blue",
                "tab:purple",
                "tab:green",
                "tab:pink",
                "tab:cyan",
                "tab:olive",
                "tab:gray",
                "tab:brown",
               ]
list_line_styles = ["solid","dotted","dashed","dashdot","solid","dotted","dashed","dashdot"]

In [ ]:
fig2, ax2 = plt.subplots(figsize=(4, 3), dpi=200)

max_time = 0.0
for idir in [1]:
    label = list_labels[idir]
    time = multi_data[label]["time"]
    
    surface_area = 4 * np.pi * np.max(multi_data[label]["XC"][1,])**2
    intensity_14 = np.max(multi_data[label]["delta_laser"]) / surface_area 
    print("Peak intensity for " + label + " is: {:4.2e} W/cm^2 \n".format(intensity_14))
    
    ax2.plot(time[1:] * 1.0e9, multi_data[label]["delta_laser"] / 1.0e12, label=label, color=list_colours[idir], linestyle=list_line_styles[0], linewidth=3)
    #ax2.plot(time[1:] * 1.0e9, multi_data[label]["delta_laser_dep"] / 1.0e12, label=" ", color=list_colours[idir], linestyle=list_line_styles[1])
    max_time = max(max_time, np.max(time[1:]))

num_profiles_per_config = 4
plasma_profile_times = np.linspace(0.5,14,num_profiles_per_config)
for iprofile in range(num_profiles_per_config):
    ax2.plot([plasma_profile_times[iprofile], plasma_profile_times[iprofile]], [-100, 1000], "r--")

ax2.set_xlabel("Time (ns)")
ax2.set_ylabel("Laser Power (TW)")
#ax2.set_xlim([0, 1.0 * max_time * 1.0e9])
ax2.set_xlim([0.0, 16])
ax2.set_ylim([0, 650.])
#ax2.legend()#loc="upper right")

fig2.savefig(plot_dir + "/" + 'taranis_pulse_and_post_processing_times.png', bbox_inches='tight')

In [ ]:

fig2, ax2 = plt.subplots(figsize=(4, 3), dpi=200)

max_time = 0.0
for idir in range(ndirs):
    label = list_labels[idir]
    time = multi_data[label]["time"]
    
    surface_area = 4 * np.pi * np.max(multi_data[label]["XC"][1,])**2
    intensity_14 = np.max(multi_data[label]["delta_laser"]) / surface_area 
    print("Peak intensity for " + label + " is: {:4.2e} W/cm^2 \n".format(intensity_14))
    
    ax2.plot(time[1:] * 1.0e9, multi_data[label]["delta_laser"] / 1.0e12, label=label, color=list_colours[idir], linestyle=list_line_styles[0], linewidth=3)
    #ax2.plot(time[1:] * 1.0e9, multi_data[label]["delta_laser_dep"] / 1.0e12, label=" ", color=list_colours[idir], linestyle=list_line_styles[1])
    max_time = max(max_time, np.max(time[1:]))
    
ax2.set_xlabel("Time (ns)")
ax2.set_ylabel("Laser Power (TW)")
#ax2.set_xlim([0, 1.0 * max_time * 1.0e9])
ax2.set_xlim([0.0, 16])
ax2.set_ylim([0, 650.])
#ax2.legend()#loc="upper right")

fig2.savefig(plot_dir + "/" + 'taranis_pulse_and_post_processing_times.png', bbox_inches='tight')

In [ ]:
aspect_ratio = (5, 4)
fig1, ax1 = plt.subplots(figsize=aspect_ratio, dpi=200)
fig2, ax2 = plt.subplots(figsize=aspect_ratio, dpi=200)

mean_pressure_estimate = 0.0
mean_pressure = 0.0
mean_difference = 0.0
n_351 = um.critical_density()

for idir in [0,1,2]:#range(ndirs):
    label = list_labels[idir]
    time = multi_data[label]["time"]

    surface_area = multi_data[label]["surface_area"]
    intensity_14 = multi_data[label]["intensity"] / 1.0e14
    pressure_estimate = multi_data[label]["pressure_estimate"] # Mbars
    
    #intensity_14 = multi_data[label]["delta_laser_dep"] / 1.0e14 / surface_area[1:]
    #pressure_estimate2 = 19.0 * (multi_data[label]["mean_deposition_density"] / n_351)**(1/9.) * (intensity_14)**(7/9)
    peak_convergence = multi_data[label]["tind_max_rhor_DT"]
    
    t_lims = np.array([0.0, 0.0])
    t_lims[0] = np.min(time[1:] / time[peak_convergence])
    t_lims[1] = np.max(time[1:] / time[peak_convergence])
    intensity_threshold = 351.0**2 / (multi_data[label]["wavelength"] * 1.0e9)**2
    max_intensity = np.max(intensity_14 / 10.0)
    print("Ratio of max intensity to LPI threshold for " + label + " is: {:.2f}".format(max_intensity / intensity_threshold))
    
    ablation_pressure = np.zeros(multi_data[label]["ntimes"])
    for tind in range(multi_data[label]["ntimes"]):
        #ablation_pressure[tind] = multi_data[label]["PT"][tind,multi_data[label]["ind_outer_surf"][tind]]
        ablation_pressure[tind] = multi_data[label]["PT"][tind,multi_data[label]["ind_ablation_front"][tind]]
    
    #plot_time = 1.5
    #tind = np.argmin(np.abs(multi_data[label]["time"][:] - plot_time * 1.0e-9))
    tind = np.argmax(pressure_estimate)
    plot_time = multi_data[label]["time"][tind]
    print(multi_data[label]["time"][tind])
    print("Intensity term: "+ label + " at time {:.2f}ns is: {:.2f} ".format(plot_time, multi_data[label]["intensity"][tind]/1.0e14))
    print("density term " + label + " at time {:.2f}ns is: {:.2f} ".format(plot_time, multi_data[label]["mean_deposition_density"][tind] / n_351))
    #print("Ablation pressure estimate "+ label + " at time {:.2f}ns is: {:.2f} Mbars".format(plot_time, pressure_estimate[tind]))
    #print("Pressure at ablation front " + label + " at time {:.2f}ns is: {:.2f} Mbars".format(plot_time, ablation_pressure[tind] / 1.0e12))
    print("Max ablation pressure estimate " + label + " is: {:.2f} Mbars".format(np.max(pressure_estimate)))
    mean_pressure_estimate += np.max(pressure_estimate)
    print("Max pressure at ablation front " + label + " is: {:.2f} Mbars".format(ablation_pressure[tind] / 1.0e12))
    mean_pressure += ablation_pressure[tind] / 1.0e12
    print("Difference,  " + label + " is: {:.2f} Mbars".format(np.abs(np.max(pressure_estimate) - ablation_pressure[tind] / 1.0e12)))
    mean_difference += np.abs(np.max(pressure_estimate) - ablation_pressure[tind] / 1.0e12)**2
    
    ax1.plot(time[1:] / time[peak_convergence], intensity_14 / 10.0, label=label, color=list_colours[idir], linestyle=list_line_styles[0], linewidth=3)
    #ax1.plot(time[1:] * 1.0e9, intensity_14 / 10.0, label=label, color=list_colours[idir], linestyle=list_line_styles[0], linewidth=3)
    ax2.plot(time[1:] / time[peak_convergence], pressure_estimate, label=label, color=list_colours[idir], linestyle=list_line_styles[0], linewidth=3)
    #ax2.plot(time[1:] / time[peak_convergence], pressure_estimate2, label=label, color=list_colours[idir], linestyle=list_line_styles[2], linewidth=3)
    
    if idir==ndirs-1:
        ax1.plot(t_lims, np.array([1.0, 1.0]) * intensity_threshold, label="LPI threshold", color=list_colours[idir], linestyle=list_line_styles[2])
        ax2.plot(t_lims, np.array([1.0, 1.0]) * 18.0 * ((intensity_threshold * 10)**(7./9.) * (351.0 / (multi_data[label]["wavelength"] * 1.0e9))**(2./9.) ), label="$P_a$ threshold", color=list_colours[idir], linestyle=list_line_styles[2])
        ax2.plot(time[1:] / time[peak_convergence], ablation_pressure[1:] / 1.0e12, label=r"$P_{a}$ hydro", color=list_colours[idir], linestyle=list_line_styles[1], linewidth=3)
    else:
        ax1.plot(t_lims, np.array([1.0, 1.0]) * intensity_threshold, color=list_colours[idir], linestyle=list_line_styles[2])
        ax2.plot(t_lims, np.array([1.0, 1.0]) * 18.0 * ((intensity_threshold * 10)**(7./9.) * (351.0 / (multi_data[label]["wavelength"] * 1.0e9))**(2./9.) ), color=list_colours[idir], linestyle=list_line_styles[2])
        ax2.plot(time[1:] / time[peak_convergence], ablation_pressure[1:] / 1.0e12, color=list_colours[idir], linestyle=list_line_styles[1], linewidth=3)
    
    #ax2.plot(time[1:] / time[peak_convergence], np.max(multi_data[label]["PT"][1:],axis=1) / 1.0e12, label=" ", color=list_colours[idir], linestyle=list_line_styles[1], linewidth=3)

xlims = [0.0,1.2]

ax1.set_xlim(xlims)
ax1.set_ylabel(r"Deposited flux ($10^{15} \mathrm{W/cm^2}$)")
ax2.set_xlabel("Time (fraction of peak convergence)")
ax1.legend(loc="upper left")
fig1.savefig(plot_dir + "/" + 'hydro_equivalent_intensity.png', bbox_inches='tight')

ylims = [0.,250]

ax2.set_xlim(xlims)
ax2.set_ylim(ylims)
ax2.set_xlabel("Time (fraction of peak convergence)")
ax2.set_ylabel("Ablation pressure (Mbars)")
ax2.legend(loc="upper left")
fig2.savefig(plot_dir + "/" + 'hydro_equivalent_ablation_pressure_estimate.png', bbox_inches='tight')

## Compare beamspots OMEGA

In [ ]:
import numpy as np
import os
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.ticker import MaxNLocator
%matplotlib inline
plt.ion();
import utils_intensity_map as uim
import utils_deck_generation as idg
import netcdf_read_write as nrw
import training_data_generation as tdg
import utils_healpy as uhp
import utils_optimizers as uopt

In [ ]:
diag_dir = "/ccc/scratch/cont002/dam/barlodun/2504a_PDDOptimization/Data/250429b_lafon351_omega_plasma_cbet"

sys_params = tdg.define_system_params(diag_dir)
sys_params["plot_file_type"] = ".png"

aspect_ratio = (5, 4)

In [ ]:
def fitness_function(dataset, opt_params):
    target_rms = opt_params["fitness_desired_rms"]
    norm_factor = opt_params["fitness_norm_factor"]
    number_of_timesteps = np.shape(dataset["rms"][:,:])[1]

    rms = np.sqrt(np.sum(dataset["rms"][:,:]**2, axis=1) / float(number_of_timesteps))
    avg_flux = np.sqrt(np.sum(dataset["avg_flux"][:,:]**2, axis=1) / float(number_of_timesteps))
    if opt_params["run_plasma_profile"]:
        target_flux = opt_params["fitness_desired_pressure_mbar"]
        indices = np.where(np.array(avg_flux) > opt_params["fitness_limit_broken_pressure_mbar"])[0]
        print("Fitness function detects broken runs: ", indices)
        avg_flux[indices] = float('nan')#0.0
    else:
        target_flux = opt_params["fitness_desired_power_per_steradian"]

    maxi_func = np.exp(-(rms/target_rms) + (avg_flux / target_flux) ** 0.25) * (avg_flux / target_flux)**0.01 * norm_factor
    #maxi_func = np.exp(-(rms/target_rms) + (avg_flux / target_flux) ** 0.25) * (avg_flux / target_flux) * norm_factor
    return maxi_func, indices

def fitness_function_time_dependant(dataset, opt_params):
    target_rms = opt_params["fitness_desired_rms"]
    norm_factor = opt_params["fitness_norm_factor"]
    number_of_timesteps = np.shape(dataset["rms"][:,:])[1]

    rms = dataset["rms"][:,:]
    avg_flux = dataset["avg_flux"][:,:]
    if opt_params["run_plasma_profile"]:
        target_flux = opt_params["fitness_desired_pressure_mbar"]
        indices = np.where(np.array(avg_flux) > opt_params["fitness_limit_broken_pressure_mbar"])[0]
        print("Fitness function detects broken runs: ", indices)
        avg_flux[indices] = float('nan')#0.0
        indices2 = np.where(np.array(avg_flux) < opt_params["fitness_limit_broken_pressure_mbar"] * 0.001)[0]
        print("Fitness function detects broken runs: ", indices2)
    else:
        target_flux = opt_params["fitness_desired_power_per_steradian"]

    maxi_func = np.exp(-(rms/target_rms) + (avg_flux / target_flux) ** 0.25) * (avg_flux / target_flux)**0.01 * norm_factor
    return maxi_func, indices

In [ ]:
dataset, dataset_params, deck_gen_params, facility_spec = idg.load_data_dicts_from_file(sys_params)
opt_params = uopt.define_optimizer_parameters(diag_dir, 0, 0, dataset_params, facility_spec, sys_params)

initial_dataset = dataset_params["num_examples"] #16#36

fitness_temporal, broken_indices = fitness_function_time_dependant(dataset, opt_params)
fitness_overall, broken_indices = fitness_function(dataset, opt_params)

labels = [None] * dataset_params["num_examples"]
for iconfig in range(dataset_params["num_examples"]):
    labels[iconfig] = r"SG={:.2f} and $R_b/R_t$={:.2f} ".format(deck_gen_params["beamspot_order"][iconfig,0], deck_gen_params["beamspot_major_radius"][iconfig,0]/dataset_params['target_radius'])

In [ ]:
ind_max_fitness = np.argmax(fitness_overall)
print(ind_max_fitness)
print(fitness_overall[ind_max_fitness])
print()

#"""
param1 = deck_gen_params["beamspot_order"][:,0]
label1 = "Super gaussian order"
param2 = deck_gen_params["beamspot_major_radius"][:,0]/dataset_params['target_radius']
label2 = r"Beam to target ratio ($R_b/R_t$)"
semilogy_bool = False
"""
param1 = deck_gen_params["bandwidth_num_spectral_lines"]
label1 = "Number spectral lines"
param2 = deck_gen_params["bandwidth_percentage_width"][:,0]
label2 = r"Bandwidth ($\delta \lambda / \lambda$)"
semilogy_bool = True
#"""

fitness_overall2 = 0.0 * param1
fitness_overall2[:np.shape(fitness_overall)[0]] = fitness_overall


In [ ]:
number_of_timesteps = np.shape(dataset["rms"])[1]

rms_overall = np.sqrt(np.sum(dataset["rms"]**2, axis=1) / float(number_of_timesteps))
avg_flux_overall = np.sqrt(np.sum(dataset["avg_flux"]**2, axis=1) / float(number_of_timesteps))
print(np.shape(dataset["rms"]), np.shape(rms_overall))

num_samples_per_param = int(np.sqrt(initial_dataset)) #int(np.sqrt(dataset_params["num_examples"]))
rms_overall_reshaped = np.reshape(rms_overall[:initial_dataset],(num_samples_per_param,num_samples_per_param))
rms_temporal_reshaped = np.reshape(dataset["rms"][:initial_dataset],(num_samples_per_param,num_samples_per_param,-1))
avg_flux_overall_reshaped = np.reshape(avg_flux_overall[:initial_dataset],(num_samples_per_param,num_samples_per_param))
avg_flux_temporal_reshaped = np.reshape(dataset["avg_flux"][:initial_dataset],(num_samples_per_param,num_samples_per_param,-1))
fitness_overall_reshaped = np.reshape(fitness_overall[:initial_dataset],(num_samples_per_param,num_samples_per_param))
fitness_temporal_reshaped = np.reshape(fitness_temporal[:initial_dataset],(num_samples_per_param,num_samples_per_param,-1))
print(np.shape(fitness_temporal_reshaped), np.shape(avg_flux_temporal_reshaped))
print(np.shape(dataset["input_parameters"]))

#X = np.reshape(dataset["input_parameters"][:,0],(num_samples_per_param,num_samples_per_param))
#Y = np.reshape(dataset["input_parameters"][:,1],(num_samples_per_param,num_samples_per_param))

X = np.reshape(param1[:initial_dataset],(num_samples_per_param,num_samples_per_param))
Y = np.reshape(param2[:initial_dataset],(num_samples_per_param,num_samples_per_param))
print(np.shape(X), np.shape(Y))
#print(rms_overall)

In [ ]:
eval_time_ind = 3
fig1, ax1 = plt.subplots(figsize=aspect_ratio, dpi=400)
fig2, ax2 = plt.subplots(figsize=aspect_ratio, dpi=400)
fig3, ax3 = plt.subplots(figsize=aspect_ratio, dpi=400)

cmap = ax1.pcolormesh(X, Y, fitness_overall_reshaped)
fig1.colorbar(cmap, ax=ax1, label="Overall fitness")

cmap = ax2.pcolormesh(X, Y, rms_overall_reshaped * 100., norm=matplotlib.colors.LogNorm(vmax=10))
fig2.colorbar(cmap, ax=ax2, label="Overall rms (%)")

cmap = ax3.pcolormesh(X, Y, avg_flux_temporal_reshaped[:,:,eval_time_ind])
fig3.colorbar(cmap, ax=ax3, label="Peak ablation pressure (Mbars)")
#cmap = ax3.pcolormesh(X, Y, avg_flux_overall_reshaped)
#fig3.colorbar(cmap, ax=ax3, label="Overall ablation pressure (Mbars)")

if semilogy_bool:
    ax1.semilogy(param1[ind_max_fitness],param2[ind_max_fitness],"rx")
    ax2.semilogy(param1[ind_max_fitness],param2[ind_max_fitness],"rx")
    ax3.semilogy(param1[ind_max_fitness],param2[ind_max_fitness],"rx")
else:
    ax1.plot(param1[ind_max_fitness],param2[ind_max_fitness],"rx")
    ax2.plot(param1[ind_max_fitness],param2[ind_max_fitness],"rx")
    ax3.plot(param1[ind_max_fitness],param2[ind_max_fitness],"rx")


ax1.set_xlabel(label1)
ax1.set_ylabel(label2)
ax2.set_xlabel(label1)
ax2.set_ylabel(label2)
ax3.set_xlabel(label1)
ax3.set_ylabel(label2)
fig1.savefig(diag_dir+"/scan_results_overall_fitness_grid" + sys_params["plot_file_type"], dpi=300, bbox_inches='tight')
fig2.savefig(diag_dir+"/scan_results_overall_rms_grid" + sys_params["plot_file_type"], dpi=300, bbox_inches='tight')
fig3.savefig(diag_dir+"/scan_results_overall_pabl_grid" + sys_params["plot_file_type"], dpi=300, bbox_inches='tight')

In [ ]:
"""
ind_max_fitness_temp = np.argmax(fitness_temporal, axis=0)
print(ind_max_fitness, ind_max_fitness_temp)
print(np.argmax(np.sqrt((fitness_temporal[:,2]**2+fitness_temporal[:,3]**2)/2.0)))

fig1, ax1 = plt.subplots(figsize=aspect_ratio, dpi=400)
fig2, ax2 = plt.subplots(figsize=aspect_ratio, dpi=400)
fig3, ax3 = plt.subplots(figsize=aspect_ratio, dpi=400)

for ind in ind_max_fitness_temp: #range(len(fitness_temporal[:,0])):#
    ax1.semilogy(dataset_params["plasma_profile_times"], fitness_temporal[ind,:], "x:", label=labels[ind])
    ax2.semilogy(dataset_params["plasma_profile_times"],dataset["rms"][ind,:] * 100, "x:", label=labels[ind])
    #ax2.semilogy(dataset_params["plasma_profile_times"],dataset["rms"][ind,:] * 100, ".", label=labels[ind])
    ax3.plot(dataset_params["plasma_profile_times"][:],dataset["avg_flux"][ind,:], "x:", label=labels[ind])
    #ax3.plot(dataset_params["plasma_profile_times"][:],dataset["avg_flux"][ind,:], ".", label=labels[ind])

xlims = [0.,30.0]
ylims = [0.01,10.0]

#ax1.set_xlim(xlims)
#ax1.set_ylim(ylims)
ax1.set_xlabel("Time (ns)")
ax1.set_ylabel("Fitness time dependant")
ax1.legend(loc="best")

xlims = [0.,30.0]
ylims = [0.,100.0]

#ax2.set_xlim(xlims)
#ax1.set_ylim(ylims)
ax2.set_xlabel("Time (ns)")
ax2.set_ylabel("RMS ablation pressure (%)")
ax2.legend(loc="best")
#fig1.savefig(root+"/"+plot_dir + "/" + 'hydro_equivalent_intensity.png', bbox_inches='tight')

#ax3.set_xlim(xlims)
ax3.set_xlabel("Time (ns)")
ax3.set_ylabel("Ablation pressure (Mbars)")
ax3.legend(loc="best")
#fig2.savefig(root+"/"+plot_dir + "/" + 'hydro_equivalent_ablation_pressure_estimate.png', bbox_inches='tight')

fig1.savefig(diag_dir+"/time_dependant_fitness" + sys_params["plot_file_type"], dpi=300, bbox_inches='tight')
fig2.savefig(diag_dir+"/time_dependant_rms" + sys_params["plot_file_type"], dpi=300, bbox_inches='tight')
fig3.savefig(diag_dir+"/time_dependant_pabl" + sys_params["plot_file_type"], dpi=300, bbox_inches='tight')
"""

## Compare beamspots M48

In [ ]:
diag_dir = "/ccc/scratch/cont002/dam/barlodun/2504a_PDDOptimization/Data/250521f_cpm48_plasma_cbet"

sys_params = tdg.define_system_params(diag_dir)
sys_params["plot_file_type"] = ".png"

aspect_ratio = (5, 4)

In [ ]:
dataset, dataset_params, deck_gen_params, facility_spec = idg.load_data_dicts_from_file(sys_params)
opt_params = uopt.define_optimizer_parameters(diag_dir, 0, 0, dataset_params, facility_spec, sys_params)

initial_dataset = dataset_params["num_examples"] #16#36

fitness_temporal, broken_indices = fitness_function_time_dependant(dataset, opt_params)
fitness_overall, _ = fitness_function(dataset, opt_params)

labels = [None] * dataset_params["num_examples"]
for iconfig in range(dataset_params["num_examples"]):
    labels[iconfig] = r"SG={:.2f} and $R_b/R_t$={:.2f} ".format(deck_gen_params["beamspot_order"][iconfig,0], deck_gen_params["beamspot_major_radius"][iconfig,0]/dataset_params['target_radius'])

In [ ]:
ind_max_fitness = np.argmax(fitness_overall)
print(ind_max_fitness)
print(fitness_overall[ind_max_fitness])
print()

#"""
param1 = deck_gen_params["beamspot_order"][:,0]
label1 = "Super gaussian order"
param2 = deck_gen_params["beamspot_major_radius"][:,0]/dataset_params['target_radius']
label2 = r"Beam to target ratio ($R_b/R_t$)"
semilogy_bool = False
"""
param1 = deck_gen_params["bandwidth_num_spectral_lines"]
label1 = "Number spectral lines"
param2 = deck_gen_params["bandwidth_percentage_width"][:,0]
label2 = r"Bandwidth ($\delta \lambda / \lambda$)"
semilogy_bool = True
#"""

fitness_overall2 = 0.0 * param1
fitness_overall2[:np.shape(fitness_overall)[0]] = fitness_overall


In [ ]:



number_of_timesteps = np.shape(dataset["rms"])[1]

rms_overall = np.sqrt(np.sum(dataset["rms"]**2, axis=1) / float(number_of_timesteps))
avg_flux_overall = np.sqrt(np.sum(dataset["avg_flux"]**2, axis=1) / float(number_of_timesteps))

dataset["rms"][broken_indices] = float("nan")
rms_overall[broken_indices] = float("nan")

num_samples_per_param = int(np.sqrt(initial_dataset)) #int(np.sqrt(dataset_params["num_examples"]))
rms_overall_reshaped = np.reshape(rms_overall[:initial_dataset],(num_samples_per_param,num_samples_per_param))
rms_temporal_reshaped = np.reshape(dataset["rms"][:initial_dataset],(num_samples_per_param,num_samples_per_param,-1))
avg_flux_overall_reshaped = np.reshape(avg_flux_overall[:initial_dataset],(num_samples_per_param,num_samples_per_param))
avg_flux_temporal_reshaped = np.reshape(dataset["avg_flux"][:initial_dataset],(num_samples_per_param,num_samples_per_param,-1))
fitness_overall_reshaped = np.reshape(fitness_overall[:initial_dataset],(num_samples_per_param,num_samples_per_param))
fitness_temporal_reshaped = np.reshape(fitness_temporal[:initial_dataset],(num_samples_per_param,num_samples_per_param,-1))
print(np.shape(fitness_temporal_reshaped), np.shape(avg_flux_temporal_reshaped))
print(np.shape(dataset["input_parameters"]))

#X = np.reshape(dataset["input_parameters"][:,0],(num_samples_per_param,num_samples_per_param))
#Y = np.reshape(dataset["input_parameters"][:,1],(num_samples_per_param,num_samples_per_param))

X = np.reshape(param1[:initial_dataset],(num_samples_per_param,num_samples_per_param))
Y = np.reshape(param2[:initial_dataset],(num_samples_per_param,num_samples_per_param))
print(np.shape(X), np.shape(Y))
#print(rms_overall)

In [ ]:
eval_time_ind=3
fig1, ax1 = plt.subplots(figsize=aspect_ratio, dpi=400)
fig2, ax2 = plt.subplots(figsize=aspect_ratio, dpi=400)
fig3, ax3 = plt.subplots(figsize=aspect_ratio, dpi=400)

cmap = ax1.pcolormesh(X, Y, fitness_overall_reshaped)
fig1.colorbar(cmap, ax=ax1, label="Overall fitness")

cmap = ax2.pcolormesh(X, Y, rms_overall_reshaped * 100., norm=matplotlib.colors.LogNorm(vmax=10))
fig2.colorbar(cmap, ax=ax2, label="Overall rms (%)")

cmap = ax3.pcolormesh(X, Y, avg_flux_temporal_reshaped[:,:,eval_time_ind])
fig3.colorbar(cmap, ax=ax3, label="Peak ablation pressure (Mbars)")

if semilogy_bool:
    ax1.semilogy(param1[ind_max_fitness],param2[ind_max_fitness],"rx")
    ax2.semilogy(param1[ind_max_fitness],param2[ind_max_fitness],"rx")
    ax3.semilogy(param1[ind_max_fitness],param2[ind_max_fitness],"rx")
else:
    ax1.plot(param1[ind_max_fitness],param2[ind_max_fitness],"rx")
    ax2.plot(param1[ind_max_fitness],param2[ind_max_fitness],"rx")
    ax3.plot(param1[ind_max_fitness],param2[ind_max_fitness],"rx")


ax1.set_xlabel(label1)
ax1.set_ylabel(label2)
ax2.set_xlabel(label1)
ax2.set_ylabel(label2)
ax3.set_xlabel(label1)
ax3.set_ylabel(label2)
fig1.savefig(diag_dir+"/scan_results_overall_fitness_grid" + sys_params["plot_file_type"], dpi=300, bbox_inches='tight')
fig2.savefig(diag_dir+"/scan_results_overall_rms_grid" + sys_params["plot_file_type"], dpi=300, bbox_inches='tight')
fig3.savefig(diag_dir+"/scan_results_overall_pabl_grid" + sys_params["plot_file_type"], dpi=300, bbox_inches='tight')

In [ ]:
ind_max_fitness_temp = np.argmax(fitness_temporal, axis=0)
print(ind_max_fitness, ind_max_fitness_temp)
print(np.argmax(np.sqrt((fitness_temporal[:,2]**2+fitness_temporal[:,3]**2)/2.0)))

fig1, ax1 = plt.subplots(figsize=aspect_ratio, dpi=400)
fig2, ax2 = plt.subplots(figsize=aspect_ratio, dpi=400)
fig3, ax3 = plt.subplots(figsize=aspect_ratio, dpi=400)

for ind in ind_max_fitness_temp: #range(len(fitness_temporal[:,0])):#
    ax1.semilogy(dataset_params["plasma_profile_times"], fitness_temporal[ind,:], "x:", label=labels[ind])
    ax2.semilogy(dataset_params["plasma_profile_times"],dataset["rms"][ind,:] * 100, "x:", label=labels[ind])
    #ax2.semilogy(dataset_params["plasma_profile_times"],dataset["rms"][ind,:] * 100, ".", label=labels[ind])
    ax3.plot(dataset_params["plasma_profile_times"][:],dataset["avg_flux"][ind,:], "x:", label=labels[ind])
    #ax3.plot(dataset_params["plasma_profile_times"][:],dataset["avg_flux"][ind,:], ".", label=labels[ind])

xlims = [0.,30.0]
ylims = [0.01,10.0]

#ax1.set_xlim(xlims)
#ax1.set_ylim(ylims)
ax1.set_xlabel("Time (ns)")
ax1.set_ylabel("Fitness time dependant")
ax1.legend(loc="best")

xlims = [0.,30.0]
ylims = [0.,100.0]

#ax2.set_xlim(xlims)
#ax1.set_ylim(ylims)
ax2.set_xlabel("Time (ns)")
ax2.set_ylabel("RMS ablation pressure (%)")
ax2.legend(loc="best")
#fig1.savefig(root+"/"+plot_dir + "/" + 'hydro_equivalent_intensity.png', bbox_inches='tight')

#ax3.set_xlim(xlims)
ax3.set_xlabel("Time (ns)")
ax3.set_ylabel("Ablation pressure (Mbars)")
ax3.legend(loc="best")
#fig2.savefig(root+"/"+plot_dir + "/" + 'hydro_equivalent_ablation_pressure_estimate.png', bbox_inches='tight')

fig1.savefig(diag_dir+"/time_dependant_fitness" + sys_params["plot_file_type"], dpi=300, bbox_inches='tight')
fig2.savefig(diag_dir+"/time_dependant_rms" + sys_params["plot_file_type"], dpi=300, bbox_inches='tight')
fig3.savefig(diag_dir+"/time_dependant_pabl" + sys_params["plot_file_type"], dpi=300, bbox_inches='tight')

## Compare illumination evaluations to hydro

In [ ]:
plot_indices = [1,2,3,4]

aspect_ratio = (5, 4)
fig1, ax1 = plt.subplots(figsize=aspect_ratio, dpi=200)
fig2, ax2 = plt.subplots(figsize=aspect_ratio, dpi=200)

mean_pressure_estimate = 0.0
mean_pressure = 0.0
mean_difference = 0.0
n_351 = um.critical_density()

for idir in plot_indices:#range(ndirs):
    label = list_labels[idir]
    time = multi_data[label]["time"]

    surface_area = multi_data[label]["surface_area"]
    intensity_14 = multi_data[label]["intensity"] / 1.0e14
    pressure_estimate = multi_data[label]["pressure_estimate"] # Mbars
    
    #intensity_14 = multi_data[label]["delta_laser_dep"] / 1.0e14 / surface_area[1:]
    #pressure_estimate2 = 19.0 * (multi_data[label]["mean_deposition_density"] / n_351)**(1/9.) * (intensity_14)**(7/9)
    peak_convergence = multi_data[label]["tind_max_rhor_DT"]
    
    t_lims = np.array([0.0, 0.0])
    t_lims[0] = np.min(time[1:] * 1.0e9)
    t_lims[1] = np.max(time[1:] * 1.0e9)
    intensity_threshold = 351.0**2 / (multi_data[label]["wavelength"] * 1.0e9)**2
    max_intensity = np.max(intensity_14 / 10.0)
    print("Ratio of max intensity to LPI threshold for " + label + " is: {:.2f}".format(max_intensity / intensity_threshold))
    
    ablation_pressure = np.zeros(multi_data[label]["ntimes"])
    for tind in range(multi_data[label]["ntimes"]):
        #ablation_pressure[tind] = multi_data[label]["PT"][tind,multi_data[label]["ind_outer_surf"][tind]]
        ablation_pressure[tind] = multi_data[label]["PT"][tind,multi_data[label]["ind_ablation_front"][tind]]
    
    #plot_time = 1.5
    #tind = np.argmin(np.abs(multi_data[label]["time"][:] - plot_time * 1.0e-9))
    tind = np.argmax(pressure_estimate)
    plot_time = multi_data[label]["time"][tind]
    print(multi_data[label]["time"][tind])
    print("Intensity term: "+ label + " at time {:.2f}ns is: {:.2f} ".format(plot_time, multi_data[label]["intensity"][tind]/1.0e14))
    print("density term " + label + " at time {:.2f}ns is: {:.2f} ".format(plot_time, multi_data[label]["mean_deposition_density"][tind] / n_351))
    #print("Ablation pressure estimate "+ label + " at time {:.2f}ns is: {:.2f} Mbars".format(plot_time, pressure_estimate[tind]))
    #print("Pressure at ablation front " + label + " at time {:.2f}ns is: {:.2f} Mbars".format(plot_time, ablation_pressure[tind] / 1.0e12))
    print("Max ablation pressure estimate " + label + " is: {:.2f} Mbars".format(np.max(pressure_estimate)))
    mean_pressure_estimate += np.max(pressure_estimate)
    print("Max pressure at ablation front " + label + " is: {:.2f} Mbars".format(ablation_pressure[tind] / 1.0e12))
    mean_pressure += ablation_pressure[tind] / 1.0e12
    print("Difference,  " + label + " is: {:.2f} Mbars".format(np.abs(np.max(pressure_estimate) - ablation_pressure[tind] / 1.0e12)))
    mean_difference += np.abs(np.max(pressure_estimate) - ablation_pressure[tind] / 1.0e12)**2
    
    ax1.plot(time[1:] * 1.0e9, intensity_14 / 10.0, label=label, color=list_colours[idir], linestyle=list_line_styles[0], linewidth=3)
    #ax1.plot(time[1:] * 1.0e9, intensity_14 / 10.0, label=label, color=list_colours[idir], linestyle=list_line_styles[0], linewidth=3)
    ax2.plot(time[1:] * 1.0e9, pressure_estimate, label=label, color=list_colours[idir], linestyle=list_line_styles[0], linewidth=3)
    #ax2.plot(time[1:] / time[peak_convergence], pressure_estimate2, label=label, color=list_colours[idir], linestyle=list_line_styles[2], linewidth=3)
    
    """
    if idir==ndirs-1:
        ax1.plot(t_lims, np.array([1.0, 1.0]) * intensity_threshold, label="LPI threshold", color=list_colours[idir], linestyle=list_line_styles[2])
        ax2.plot(t_lims, np.array([1.0, 1.0]) * 18.0 * ((intensity_threshold * 10)**(7./9.) * (351.0 / (multi_data[label]["wavelength"] * 1.0e9))**(2./9.) ), label="$P_a$ threshold", color=list_colours[idir], linestyle=list_line_styles[2])
        ax2.plot(time[1:] * 1.0e9, ablation_pressure[1:] / 1.0e12, label=r"$P_{a}$ hydro", color=list_colours[idir], linestyle=list_line_styles[1], linewidth=3)
    else:
        ax1.plot(t_lims, np.array([1.0, 1.0]) * intensity_threshold, color=list_colours[idir], linestyle=list_line_styles[2])
        ax2.plot(t_lims, np.array([1.0, 1.0]) * 18.0 * ((intensity_threshold * 10)**(7./9.) * (351.0 / (multi_data[label]["wavelength"] * 1.0e9))**(2./9.) ), color=list_colours[idir], linestyle=list_line_styles[2])
        ax2.plot(time[1:] * 1.0e9, ablation_pressure[1:] / 1.0e12, color=list_colours[idir], linestyle=list_line_styles[1], linewidth=3)
    """
    #ax2.plot(time[1:] / time[peak_convergence], np.max(multi_data[label]["PT"][1:],axis=1) / 1.0e12, label=" ", color=list_colours[idir], linestyle=list_line_styles[1], linewidth=3)

for ind in ind_max_fitness_temp: #range(len(fitness_temporal[:,0])):#
    ax2.plot(dataset_params["plasma_profile_times"][:],dataset["avg_flux"][ind,:], "x:", label=labels[ind])
    
xlims = [0.0,1.2]

#ax1.set_xlim(xlims)
ax1.set_ylabel(r"Deposited flux ($10^{15} \mathrm{W/cm^2}$)")
ax2.set_xlabel("Time (fraction of peak convergence)")
ax1.legend(loc="upper left")
fig1.savefig(plot_dir + "/" + 'hydro_equivalent_intensity.png', bbox_inches='tight')

ylims = [0.,250]

#ax2.set_xlim(xlims)
ax2.set_ylim(ylims)
ax2.set_xlabel("Time (fraction of peak convergence)")
ax2.set_ylabel("Ablation pressure (Mbars)")
ax2.legend(loc="upper left")
fig2.savefig(plot_dir + "/" + 'hydro_equivalent_ablation_pressure_estimate.png', bbox_inches='tight')

In [ ]:

fig1, ax1 = plt.subplots(figsize=(5, 4), dpi=200)
fig2, ax2 = plt.subplots(figsize=(5, 4), dpi=200)
fig3, ax3 = plt.subplots(figsize=(5, 4), dpi=200)

plot_markers = ("o", "s", "^", "*", "X")

for idir in plot_indices:
    label = list_labels[idir]
    #label2 = list_labels2[idir]
    time = multi_data[label]["time"]
    gain_emitted_laser = multi_data[label]["energy_fusion_emitted"][-1] / multi_data[list_labels[1]]["energy_laser_emitted"][-1]
    #gain_emitted_laser = multi_data[label]["energy_fusion_emitted"][-1] / multi_data[label]["energy_laser_emitted"][-1]
    ax1.plot(multi_data[label]["lawson_criteria_chang2010"], gain_emitted_laser, "o", color=list_colours[idir], label=label, markersize=12)
    ax2.plot(np.max(multi_data[label]["pressure_estimate"]), gain_emitted_laser, "o", color=list_colours[idir], label=label, markersize=12)
    #ax2.semilogy(np.max(multi_data[label]["pressure_estimate"]), gain_emitted_laser, "o", color=list_colours[idir], label=label, markersize=12)
    ax3.plot(multi_data[label]["energy_laser_deposited"][-1]/1.0e6, gain_emitted_laser, "o", color=list_colours[idir], label=label, markersize=12)
    #ax2.plot(multi_data[label]["beam_fwhm"]/2.0*1.0e6, np.max(multi_data[label]["pressure_estimate"]), "o", color=list_colours[idir], label=label)

xlims = [0.,1.8]
ylims = [0.,80]
#ax1.set_xlim(xlims)
#ax1.set_ylim(ylims)
ax1.set_ylabel(r"Gain ($E_f / E_L$)")
ax1.set_xlabel(r"Ignition Criterion $\chi$")
ax1.legend(loc="lower right")
fig1.savefig(plot_dir + "/" + 'implosion_performance.png', bbox_inches='tight')

xlims = [70.,300.]
ylims = [0.,80]
#ax2.set_xlim(xlims)
#ax2.set_ylim(ylims)
ax2.set_ylabel(r"Gain ($E_f / E_L$)")
ax2.set_xlabel("Peak ablation pressure (Mbars)")
ax2.legend(loc="lower right")
fig2.savefig(plot_dir + "/" + 'implosion_performance_ablation_pressure.png', bbox_inches='tight')


xlims = [0.,1.8]
ylims = [0.,80]
#ax1.set_xlim(xlims)
#ax1.set_ylim(ylims)
ax3.set_ylabel(r"Gain ($E_f / E_L$)")
ax3.set_xlabel(r"Laser energy absorbed")
ax3.legend(loc="lower right")
fig3.savefig(plot_dir + "/" + 'implosion_performance_laser_energy.png', bbox_inches='tight')


## Varying bandwidth at OMEGA

In [ ]:
import numpy as np
import os
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.ticker import MaxNLocator
%matplotlib inline
plt.ion();
import utils_intensity_map as uim
import utils_deck_generation as idg
import netcdf_read_write as nrw
import training_data_generation as tdg
import utils_healpy as uhp
import utils_optimizers as uopt

def fitness_function(dataset, opt_params):
    target_rms = opt_params["fitness_desired_rms"]
    norm_factor = opt_params["fitness_norm_factor"]
    number_of_timesteps = np.shape(dataset["rms"][:,:])[1]

    rms = np.sqrt(np.sum(dataset["rms"][:,:]**2, axis=1) / float(number_of_timesteps))
    avg_flux = np.sqrt(np.sum(dataset["avg_flux"][:,:]**2, axis=1) / float(number_of_timesteps))
    if opt_params["run_plasma_profile"]:
        target_flux = opt_params["fitness_desired_pressure_mbar"]
        indices = np.where(np.array(avg_flux) > opt_params["fitness_limit_broken_pressure_mbar"])[0]
        #print("Fitness function detects broken runs: ", indices)
        avg_flux[indices] = 0.0
    else:
        target_flux = opt_params["fitness_desired_power_per_steradian"]

    maxi_func = np.exp(-(rms/target_rms) + (avg_flux / target_flux) ** 0.33) * (avg_flux / target_flux)**0.01 * norm_factor
    #maxi_func = np.exp(-(rms/target_rms) + (avg_flux / target_flux) ** 0.25) * (avg_flux / target_flux)**0.01 * norm_factor
    return maxi_func

def fitness_function_time_dependant(dataset, opt_params):
    target_rms = opt_params["fitness_desired_rms"]
    norm_factor = opt_params["fitness_norm_factor"]
    number_of_timesteps = np.shape(dataset["rms"][:,:])[1]

    rms = dataset["rms"][:,:]
    avg_flux = dataset["avg_flux"][:,:]
    if opt_params["run_plasma_profile"]:
        target_flux = opt_params["fitness_desired_pressure_mbar"]
        indices1 = np.where(np.array(avg_flux) > opt_params["fitness_limit_broken_pressure_mbar"])[0]
        avg_flux[indices1] = 0.0
        indices = np.where(np.array(avg_flux) < opt_params["fitness_limit_broken_pressure_mbar"] * 0.001)[0]
        print("Fitness function detects broken runs: ", indices1, indices)
    else:
        target_flux = opt_params["fitness_desired_power_per_steradian"]

    maxi_func = np.exp(-(rms/target_rms) + (avg_flux / target_flux) ** 0.25) * (avg_flux / target_flux)**0.01 * norm_factor
    return maxi_func

In [ ]:
polaris_list_labels = [r"$\Omega$60 no CBET",
               r"$\Omega$60 CBET",
               r"$\Omega$60 varying $\Delta \lambda$",]
polaris_list_dirs = ["/ccc/scratch/cont002/dam/barlodun/2504a_PDDOptimization/Data/250429c_lafon351_omega_plasma_more",
             "/ccc/scratch/cont002/dam/barlodun/2504a_PDDOptimization/Data/250429b_lafon351_omega_plasma_cbet",
             "/ccc/scratch/cont002/dam/barlodun/2504a_PDDOptimization/Data/250507a_lafon351_omega_plasma_bandwidth_14ns",]

plot_file_type = ".png"

aspect_ratio = (5, 4)


In [ ]:

list_line_styles = ["solid","dotted","dashed","dashdot","solid","dotted","dashed","dashdot","solid","dotted","dashed","dashdot","solid","dotted","dashed","dashdot"]
list_point_styles = matplotlib.lines.Line2D.markers
print(list_point_styles)

In [ ]:
ndirs = len(polaris_list_labels)
sys_params = {}
dataset = {}
dataset_params = {}
deck_gen_params = {}
facility_spec = {}
opt_params = {}
fitness_temporal = {}
fitness_overall = {}
label_beam_config = {}

for idir in range(ndirs):
    label = polaris_list_labels[idir]
    loc_dir = polaris_list_dirs[idir]
    sys_params[label] = {}
    dataset[label] = {}
    dataset_params[label] = {}
    deck_gen_params[label] = {}
    facility_spec[label] = {}
    opt_params[label] = {}
    fitness_temporal[label] = {}
    fitness_overall[label] = {}

    print(label, loc_dir)
    sys_params[label] = tdg.define_system_params(loc_dir)
    dataset[label], dataset_params[label], deck_gen_params[label], facility_spec[label] = idg.load_data_dicts_from_file(sys_params[label])
    opt_params[label] = uopt.define_optimizer_parameters(loc_dir, 0, 0, dataset_params[label], facility_spec[label], sys_params[label])
    #opt_params["fitness_limit_broken_pressure_mbar"] = 1000.0
    print(opt_params[label]["run_plasma_profile"])
    print(dataset[label]["rms"][0])
    print(dataset[label]["avg_flux"][0])

    fitness_temporal[label] = fitness_function_time_dependant(dataset[label], opt_params[label])
    fitness_overall[label] = fitness_function(dataset[label], opt_params[label])

    label_beam_config[label] = [None] * dataset_params[label]["num_examples"]
    for iconfig in range(dataset_params[label]["num_examples"]):
        label_beam_config[label][iconfig] = r"SG={:.2f} and $R_b/R_t$={:.2f} ".format(deck_gen_params[label]["beamspot_order"][iconfig,0], deck_gen_params[label]["beamspot_major_radius"][iconfig,0]/dataset_params[label]['target_radius'])
    label_beam_config[label] = np.array(label_beam_config[label])

In [ ]:

ind_max_fitness = {}
ind_max_fitness_list = {}
ind_max_fitness_temp = {}
ind_min_rms_early_time = {}
rms_overall = {}

for idir in range(ndirs):
    label = polaris_list_labels[idir]
    number_of_timesteps = np.shape(dataset[label]["rms"])[1]
    
    ind_max_fitness[label] = np.argmax(fitness_overall[label])
    ind_max_fitness_list[label] = np.array(np.where(fitness_overall[label]>0.5)[0], dtype='int')
    print(label, len(ind_max_fitness_list[label]), np.max(fitness_overall[label]))
    rms_overall[label] = np.sqrt(np.sum(dataset[label]["rms"][:,:]**2, axis=1) / float(number_of_timesteps))
    
    facility_best_bs_rms_overall = rms_overall[label][ind_max_fitness_list[label]]
    
    ind_max_fitness_temp[label] = np.array(np.argmax(fitness_temporal[label][:,-1]), dtype='int')
    facility_best_late_time_bs_rms = dataset[label]["rms"][ind_max_fitness_temp[label],-1]
    facility_best_late_time_bs_rms_early_time = dataset[label]["rms"][ind_max_fitness_temp[label],0]
    
    """
    if label == "CPM72 193nm":
        facility_best_bs_ablation_pressure_late_time = dataset[label]["avg_flux"][ind_max_fitness_list[label],-2]
        facility_best_late_time_bs_ablation_pressure = dataset[label]["avg_flux"][ind_max_fitness_temp[label],-2]
    else:
    """
    facility_best_bs_ablation_pressure_late_time = dataset[label]["avg_flux"][ind_max_fitness_list[label],-1]
    facility_best_late_time_bs_ablation_pressure = dataset[label]["avg_flux"][ind_max_fitness_temp[label],-1]
    
    facility_best_early_time_bs_rms = np.min(dataset[label]["rms"][:,0][np.where(dataset[label]["avg_flux"][:,0]>0.0001)[0]])
    ind_min_rms_early_time[label] = np.where(dataset[label]["rms"][:,0]==facility_best_early_time_bs_rms)[0]
    facility_best_early_time_bs_pabl = dataset[label]["avg_flux"][ind_min_rms_early_time[label],0]
    
    sort_key = np.argsort(facility_best_bs_rms_overall)
    line_valx = [facility_best_bs_rms_overall[sort_key[0]]]
    line_valy = [facility_best_bs_ablation_pressure_late_time[sort_key[0]]]
    for ind in range(len(sort_key)):
        if facility_best_bs_ablation_pressure_late_time[sort_key[ind]]>line_valy[-1]:
            line_valx.append(facility_best_bs_rms_overall[sort_key[ind]])
            line_valy.append(facility_best_bs_ablation_pressure_late_time[sort_key[ind]])
    line_valx = np.array(line_valx)
    line_valy = np.array(line_valy)

In [ ]:
fig1, ax1 = plt.subplots(figsize=aspect_ratio, dpi=400)

ind_max_fitness = {}
ind_max_fitness_list = {}
ind_max_fitness_temp = {}
ind_min_rms_early_time = {}


for idir in range(2):
    label = polaris_list_labels[idir]
    number_of_timesteps = np.shape(dataset[label]["rms"])[1]
    ind_max_fitness[label] = np.argmax(fitness_overall[label])
    ind_max_fitness_list[label] = np.array(np.where(fitness_overall[label]>0.5)[0], dtype='int')
    print(label, len(ind_max_fitness_list[label]), np.max(fitness_overall[label]), list_point_styles[idir+3])
    rms_overall[label] = np.sqrt(np.sum(dataset[label]["rms"][:,:]**2, axis=1) / float(number_of_timesteps))
    
    facility_best_bs_rms_overall = rms_overall[label][ind_max_fitness_list[label]]
    facility_best_bs_ablation_pressure_late_time = dataset[label]["avg_flux"][ind_max_fitness_list[label],-1]

    ind_max_fitness_temp[label] = np.array(np.argmax(fitness_temporal[label][:,-1]), dtype='int')
    facility_best_late_time_bs_rms = dataset[label]["rms"][ind_max_fitness_temp[label],-1]
    facility_best_late_time_bs_rms_early_time = dataset[label]["rms"][ind_max_fitness_temp[label],0]
    facility_best_late_time_bs_ablation_pressure = dataset[label]["avg_flux"][ind_max_fitness_temp[label],-1]
    
    facility_best_early_time_bs_rms = np.min(dataset[label]["rms"][:,0][np.where(dataset[label]["avg_flux"][:,0]>0.0001)[0]])
    ind_min_rms_early_time[label] = np.where(dataset[label]["rms"][:,0]==facility_best_early_time_bs_rms)[0]
    facility_best_early_time_bs_pabl = dataset[label]["avg_flux"][ind_min_rms_early_time[label],0]

    #ax1.semilogx(facility_best_bs_rms_overall*100, facility_best_bs_ablation_pressure_late_time, "o",
    #             markersize=5, label=label, color=list_colours[idir])
    
    sort_key = np.argsort(facility_best_bs_rms_overall)
    line_valx = [facility_best_bs_rms_overall[sort_key[0]]]
    line_valy = [facility_best_bs_ablation_pressure_late_time[sort_key[0]]]
    #print(line_valy)
    for ind in range(len(sort_key)):
        if facility_best_bs_ablation_pressure_late_time[sort_key[ind]]>line_valy[-1]:
            line_valx.append(facility_best_bs_rms_overall[sort_key[ind]])
            line_valy.append(facility_best_bs_ablation_pressure_late_time[sort_key[ind]])
    line_valx = np.array(line_valx)
    line_valy = np.array(line_valy)
    ax1.semilogx(line_valx * 100.0, line_valy, color=list_colours[idir], label=label)

idir = 2
label = polaris_list_labels[idir]
indices_most_spectral_lines = np.where(deck_gen_params[label]["bandwidth_num_spectral_lines"]==20)[0]
rms_overall_most_spectral_lines = rms_overall[label][indices_most_spectral_lines]
ablation_pressure_late_time_most_spectral_lines = dataset[label]["avg_flux"][indices_most_spectral_lines,-1]

ax1.semilogx(rms_overall_most_spectral_lines*100, ablation_pressure_late_time_most_spectral_lines, "o",
             markersize=5, label=label, color=list_colours[idir])

num_ind = len(indices_most_spectral_lines)
label_bandwidth_beam_config = [None] * num_ind
for ind in range(num_ind):#range(num_ind):
    label_bandwidth_beam_config[ind] = r"$\Delta \lambda $={:.1f}%".format(deck_gen_params[label]["bandwidth_percentage_width"][indices_most_spectral_lines[ind],0])
    print(num_ind, ind)
    print(rms_overall_most_spectral_lines[ind]*100, ablation_pressure_late_time_most_spectral_lines[ind],
             label_bandwidth_beam_config[ind])
    horizontalalignment = "left"
    if ind == (num_ind-1):
        horizontalalignment = "right"
    ax1.text(rms_overall_most_spectral_lines[ind]*100, ablation_pressure_late_time_most_spectral_lines[ind],
             label_bandwidth_beam_config[ind], fontsize=10, horizontalalignment=horizontalalignment)
ax1.legend(loc="lower right")

ax1.set_xlim([0.1,20])
ax1.set_ylim([0,250])
ax1.set_xlabel("RMS ablation pressure (% mean)")
ax1.set_ylabel("Ablation pressure (Mbars)")
fig1.savefig(loc_dir+"/../../"+sys_params[label]["figure_location"]+"/facility_comparison_pabl_v_rms_193nm_bandwidth" + plot_file_type, dpi=300, bbox_inches='tight')


## Varying beamspots for 1% bandwidth CPM48

In [ ]:
polaris_list_labels = ["CPM48 no CBET",
               "CPM48 CBET",
               r"CPM48 $\Delta \lambda$ = 1%",]
polaris_list_dirs = ["/ccc/scratch/cont002/dam/barlodun/2504a_PDDOptimization/Data/250506i_cpm48_plasma",
             "/ccc/scratch/cont002/dam/barlodun/2504a_PDDOptimization/Data/250521f_cpm48_plasma_cbet",
             "/ccc/scratch/cont002/dam/barlodun/2504a_PDDOptimization/Data/250513g_cpm48_plasma_vary_beamspot_with_bandwidth_partial_fail",]


In [ ]:
ndirs = len(polaris_list_labels)
sys_params = {}
dataset = {}
dataset_params = {}
deck_gen_params = {}
facility_spec = {}
opt_params = {}
fitness_temporal = {}
fitness_overall = {}
label_beam_config = {}

for idir in range(ndirs):
    label = polaris_list_labels[idir]
    loc_dir = polaris_list_dirs[idir]
    sys_params[label] = {}
    dataset[label] = {}
    dataset_params[label] = {}
    deck_gen_params[label] = {}
    facility_spec[label] = {}
    opt_params[label] = {}
    fitness_temporal[label] = {}
    fitness_overall[label] = {}

    print(label, loc_dir)
    sys_params[label] = tdg.define_system_params(loc_dir)
    dataset[label], dataset_params[label], deck_gen_params[label], facility_spec[label] = idg.load_data_dicts_from_file(sys_params[label])
    opt_params[label] = uopt.define_optimizer_parameters(loc_dir, 0, 0, dataset_params[label], facility_spec[label], sys_params[label])
    #opt_params["fitness_limit_broken_pressure_mbar"] = 1000.0
    print(opt_params[label]["run_plasma_profile"])
    print(dataset[label]["rms"][0])
    print(dataset[label]["avg_flux"][0])

    fitness_temporal[label] = fitness_function_time_dependant(dataset[label], opt_params[label])
    fitness_overall[label] = fitness_function(dataset[label], opt_params[label])

    label_beam_config[label] = [None] * dataset_params[label]["num_examples"]
    for iconfig in range(dataset_params[label]["num_examples"]):
        label_beam_config[label][iconfig] = r"SG={:.2f} and $R_b/R_t$={:.2f} ".format(deck_gen_params[label]["beamspot_order"][iconfig,0], deck_gen_params[label]["beamspot_major_radius"][iconfig,0]/dataset_params[label]['target_radius'])
    label_beam_config[label] = np.array(label_beam_config[label])

In [ ]:
fig1, ax1 = plt.subplots(figsize=aspect_ratio, dpi=400)

ind_max_fitness = {}
ind_max_fitness_list = {}
ind_max_fitness_temp = {}
ind_min_rms_early_time = {}
rms_overall = {}

for idir in range(ndirs):
    label = polaris_list_labels[idir]
    number_of_timesteps = np.shape(dataset[label]["rms"])[1]
    
    ind_max_fitness[label] = np.argmax(fitness_overall[label])
    ind_max_fitness_list[label] = np.array(np.where(fitness_overall[label]>0.5)[0], dtype='int')
    print(label, len(ind_max_fitness_list[label]), np.max(fitness_overall[label]))
    rms_overall[label] = np.sqrt(np.sum(dataset[label]["rms"][:,:]**2, axis=1) / float(number_of_timesteps))
    
    facility_best_bs_rms_overall = rms_overall[label][ind_max_fitness_list[label]]
    
    ind_max_fitness_temp[label] = np.array(np.argmax(fitness_temporal[label][:,-1]), dtype='int')
    facility_best_late_time_bs_rms = dataset[label]["rms"][ind_max_fitness_temp[label],-1]
    facility_best_late_time_bs_rms_early_time = dataset[label]["rms"][ind_max_fitness_temp[label],0]
    
    """
    if label == "CPM72 193nm":
        facility_best_bs_ablation_pressure_late_time = dataset[label]["avg_flux"][ind_max_fitness_list[label],-2]
        facility_best_late_time_bs_ablation_pressure = dataset[label]["avg_flux"][ind_max_fitness_temp[label],-2]
    else:
    """
    facility_best_bs_ablation_pressure_late_time = dataset[label]["avg_flux"][ind_max_fitness_list[label],-1]
    facility_best_late_time_bs_ablation_pressure = dataset[label]["avg_flux"][ind_max_fitness_temp[label],-1]
    
    facility_best_early_time_bs_rms = np.min(dataset[label]["rms"][:,0][np.where(dataset[label]["avg_flux"][:,0]>0.0001)[0]])
    ind_min_rms_early_time[label] = np.where(dataset[label]["rms"][:,0]==facility_best_early_time_bs_rms)[0]
    facility_best_early_time_bs_pabl = dataset[label]["avg_flux"][ind_min_rms_early_time[label],0]

    #print(label, facility_best_bs_rms_overall*100, facility_best_bs_ablation_pressure_late_time)
    ax1.semilogx(facility_best_bs_rms_overall*100, facility_best_bs_ablation_pressure_late_time, "o",
                markersize=5, color=list_colours[idir])#, label=label)
    ax1.semilogx(facility_best_late_time_bs_rms*100, facility_best_late_time_bs_ablation_pressure, "x",
                markersize=5, color=list_colours[idir])
    ax1.semilogx(facility_best_early_time_bs_rms*100, facility_best_early_time_bs_pabl, "^",
                markersize=5, color=list_colours[idir])
    
    sort_key = np.argsort(facility_best_bs_rms_overall)
    line_valx = [facility_best_bs_rms_overall[sort_key[0]]]
    line_valy = [facility_best_bs_ablation_pressure_late_time[sort_key[0]]]
    for ind in range(len(sort_key)):
        if facility_best_bs_ablation_pressure_late_time[sort_key[ind]]>line_valy[-1]:
            line_valx.append(facility_best_bs_rms_overall[sort_key[ind]])
            line_valy.append(facility_best_bs_ablation_pressure_late_time[sort_key[ind]])
    line_valx = np.array(line_valx)
    line_valy = np.array(line_valy)
    ax1.semilogx(line_valx * 100.0, line_valy, color=list_colours[idir], label=label)
    """ 
    ax1.text(rms_overall[label][ind_max_fitness[label]]*100, dataset[label]["avg_flux"][ind_max_fitness[label],-1],
                     label_beam_config[label][ind_max_fitness[label]], fontsize=5, rotation=45)
    ax1.text(facility_best_late_time_bs_rms*100, facility_best_late_time_bs_ablation_pressure,
                     label_beam_config[label][ind_max_fitness_temp[label]], fontsize=5, rotation=45)
    ax1.text(dataset[label]["rms"][ind_min_rms_early_time[label],0]*100, dataset[label]["avg_flux"][ind_min_rms_early_time[label],0],
                     label_beam_config[label][ind_min_rms_early_time[label]][0], fontsize=5, rotation=45)
    
    #"""
    """
    for ind in range(len(ind_max_fitness[label])):
        if (ind % 4 == 0):
            ax1.text(facility_best_bs_rms_overall[label][ind]*100, facility_best_bs_ablation_pressure_late_time[label][ind],
                     label_beam_config[label][ind_max_fitness[label][ind]], fontsize=5)
    """

ax1.legend(loc="lower right")

ax1.set_xlim([0.1,20])
ax1.set_ylim([0,250])
ax1.set_xlabel("RMS ablation pressure (% mean)")
ax1.set_ylabel("Ablation pressure (Mbars)")
fig1.savefig(loc_dir+"/../../"+sys_params[label]["figure_location"]+"/facility_comparison_pabl_and_rms" + plot_file_type, dpi=300, bbox_inches='tight')


## Performance with bandwidth

In [ ]:
plot_indices = [1,2,5,6,7]

aspect_ratio = (5, 4)
fig1, ax1 = plt.subplots(figsize=aspect_ratio, dpi=200)
fig2, ax2 = plt.subplots(figsize=aspect_ratio, dpi=200)

mean_pressure_estimate = 0.0
mean_pressure = 0.0
mean_difference = 0.0
n_351 = um.critical_density()

for idir in plot_indices:#range(ndirs):
    label = list_labels[idir]
    time = multi_data[label]["time"]

    surface_area = multi_data[label]["surface_area"]
    intensity_14 = multi_data[label]["intensity"] / 1.0e14
    pressure_estimate = multi_data[label]["pressure_estimate"] # Mbars
    
    #intensity_14 = multi_data[label]["delta_laser_dep"] / 1.0e14 / surface_area[1:]
    #pressure_estimate2 = 19.0 * (multi_data[label]["mean_deposition_density"] / n_351)**(1/9.) * (intensity_14)**(7/9)
    peak_convergence = multi_data[label]["tind_max_rhor_DT"]
    
    t_lims = np.array([0.0, 0.0])
    t_lims[0] = np.min(time[1:] * 1.0e9)
    t_lims[1] = np.max(time[1:] * 1.0e9)
    intensity_threshold = 351.0**2 / (multi_data[label]["wavelength"] * 1.0e9)**2
    max_intensity = np.max(intensity_14 / 10.0)
    print("Ratio of max intensity to LPI threshold for " + label + " is: {:.2f}".format(max_intensity / intensity_threshold))
    
    ablation_pressure = np.zeros(multi_data[label]["ntimes"])
    for tind in range(multi_data[label]["ntimes"]):
        #ablation_pressure[tind] = multi_data[label]["PT"][tind,multi_data[label]["ind_outer_surf"][tind]]
        ablation_pressure[tind] = multi_data[label]["PT"][tind,multi_data[label]["ind_ablation_front"][tind]]
    
    #plot_time = 1.5
    #tind = np.argmin(np.abs(multi_data[label]["time"][:] - plot_time * 1.0e-9))
    tind = np.argmax(pressure_estimate)
    plot_time = multi_data[label]["time"][tind]
    print(multi_data[label]["time"][tind])
    print("Intensity term: "+ label + " at time {:.2f}ns is: {:.2f} ".format(plot_time, multi_data[label]["intensity"][tind]/1.0e14))
    print("density term " + label + " at time {:.2f}ns is: {:.2f} ".format(plot_time, multi_data[label]["mean_deposition_density"][tind] / n_351))
    #print("Ablation pressure estimate "+ label + " at time {:.2f}ns is: {:.2f} Mbars".format(plot_time, pressure_estimate[tind]))
    #print("Pressure at ablation front " + label + " at time {:.2f}ns is: {:.2f} Mbars".format(plot_time, ablation_pressure[tind] / 1.0e12))
    print("Max ablation pressure estimate " + label + " is: {:.2f} Mbars".format(np.max(pressure_estimate)))
    mean_pressure_estimate += np.max(pressure_estimate)
    print("Max pressure at ablation front " + label + " is: {:.2f} Mbars".format(ablation_pressure[tind] / 1.0e12))
    mean_pressure += ablation_pressure[tind] / 1.0e12
    print("Difference,  " + label + " is: {:.2f} Mbars".format(np.abs(np.max(pressure_estimate) - ablation_pressure[tind] / 1.0e12)))
    mean_difference += np.abs(np.max(pressure_estimate) - ablation_pressure[tind] / 1.0e12)**2
    
    ax1.plot(time[1:] * 1.0e9, intensity_14 / 10.0, label=label, color=list_colours[idir], linestyle=list_line_styles[0], linewidth=3)
    #ax1.plot(time[1:] * 1.0e9, intensity_14 / 10.0, label=label, color=list_colours[idir], linestyle=list_line_styles[0], linewidth=3)
    ax2.plot(time[1:] * 1.0e9, pressure_estimate, label=label, color=list_colours[idir], linestyle=list_line_styles[0], linewidth=3)
    #ax2.plot(time[1:] / time[peak_convergence], pressure_estimate2, label=label, color=list_colours[idir], linestyle=list_line_styles[2], linewidth=3)
    
    """
    if idir==ndirs-1:
        ax1.plot(t_lims, np.array([1.0, 1.0]) * intensity_threshold, label="LPI threshold", color=list_colours[idir], linestyle=list_line_styles[2])
        ax2.plot(t_lims, np.array([1.0, 1.0]) * 18.0 * ((intensity_threshold * 10)**(7./9.) * (351.0 / (multi_data[label]["wavelength"] * 1.0e9))**(2./9.) ), label="$P_a$ threshold", color=list_colours[idir], linestyle=list_line_styles[2])
        ax2.plot(time[1:] * 1.0e9, ablation_pressure[1:] / 1.0e12, label=r"$P_{a}$ hydro", color=list_colours[idir], linestyle=list_line_styles[1], linewidth=3)
    else:
        ax1.plot(t_lims, np.array([1.0, 1.0]) * intensity_threshold, color=list_colours[idir], linestyle=list_line_styles[2])
        ax2.plot(t_lims, np.array([1.0, 1.0]) * 18.0 * ((intensity_threshold * 10)**(7./9.) * (351.0 / (multi_data[label]["wavelength"] * 1.0e9))**(2./9.) ), color=list_colours[idir], linestyle=list_line_styles[2])
        ax2.plot(time[1:] * 1.0e9, ablation_pressure[1:] / 1.0e12, color=list_colours[idir], linestyle=list_line_styles[1], linewidth=3)
    """
    #ax2.plot(time[1:] / time[peak_convergence], np.max(multi_data[label]["PT"][1:],axis=1) / 1.0e12, label=" ", color=list_colours[idir], linestyle=list_line_styles[1], linewidth=3)

idir = 2
label = polaris_list_labels[idir]

ind_max_fitness_temp[label] = np.array(np.argmax(fitness_temporal[label], axis=0), dtype='int')
#ind_max_fitness_temp = np.argmax(fitness_temporal, axis=0)
print(label, ind_max_fitness_temp[label])
for ind in ind_max_fitness_temp[label]: #range(len(fitness_temporal[:,0])):#
    ax2.plot(dataset_params[label]["plasma_profile_times"][:],dataset[label]["avg_flux"][ind,:], "x:", label=labels[ind])
    
xlims = [0.0,1.2]

#ax1.set_xlim(xlims)
ax1.set_ylabel(r"Deposited flux ($10^{15} \mathrm{W/cm^2}$)")
ax2.set_xlabel("Time (fraction of peak convergence)")
ax1.legend(loc="upper left")
fig1.savefig(plot_dir + "/" + 'hydro_equivalent_intensity.png', bbox_inches='tight')

ylims = [0.,250]

#ax2.set_xlim(xlims)
ax2.set_ylim(ylims)
ax2.set_xlabel("Time (fraction of peak convergence)")
ax2.set_ylabel("Ablation pressure (Mbars)")
ax2.legend(loc="upper left")
fig2.savefig(plot_dir + "/" + 'hydro_equivalent_ablation_pressure_estimate.png', bbox_inches='tight')

In [ ]:

fig1, ax1 = plt.subplots(figsize=(5, 4), dpi=200)
fig2, ax2 = plt.subplots(figsize=(5, 4), dpi=200)
fig3, ax3 = plt.subplots(figsize=(5, 4), dpi=200)

plot_markers = ("o", "s", "^", "*", "X")

for idir in plot_indices:
    label = list_labels[idir]
    #label2 = list_labels2[idir]
    time = multi_data[label]["time"]
    gain_emitted_laser = multi_data[label]["energy_fusion_emitted"][-1] / multi_data[list_labels[1]]["energy_laser_emitted"][-1]
    #gain_emitted_laser = multi_data[label]["energy_fusion_emitted"][-1] / multi_data[label]["energy_laser_emitted"][-1]
    ax1.plot(multi_data[label]["lawson_criteria_chang2010"], gain_emitted_laser, "o", color=list_colours[idir], label=label, markersize=12)
    ax2.plot(np.max(multi_data[label]["pressure_estimate"]), gain_emitted_laser, "o", color=list_colours[idir], label=label, markersize=12)
    #ax2.semilogy(np.max(multi_data[label]["pressure_estimate"]), gain_emitted_laser, "o", color=list_colours[idir], label=label, markersize=12)
    ax3.plot(multi_data[label]["energy_laser_deposited"][-1]/1.0e6, gain_emitted_laser, "o", color=list_colours[idir], label=label, markersize=12)
    #ax2.plot(multi_data[label]["beam_fwhm"]/2.0*1.0e6, np.max(multi_data[label]["pressure_estimate"]), "o", color=list_colours[idir], label=label)

xlims = [0.,1.8]
ylims = [0.,80]
#ax1.set_xlim(xlims)
#ax1.set_ylim(ylims)
ax1.set_ylabel(r"Gain ($E_f / E_L$)")
ax1.set_xlabel(r"Ignition Criterion $\chi$")
ax1.legend(loc="lower right")
fig1.savefig(plot_dir + "/" + 'implosion_performance.png', bbox_inches='tight')

xlims = [70.,300.]
ylims = [0.,80]
#ax2.set_xlim(xlims)
#ax2.set_ylim(ylims)
ax2.set_ylabel(r"Gain ($E_f / E_L$)")
ax2.set_xlabel("Peak ablation pressure (Mbars)")
ax2.legend(loc="lower right")
fig2.savefig(plot_dir + "/" + 'implosion_performance_ablation_pressure.png', bbox_inches='tight')


xlims = [0.,1.8]
ylims = [0.,80]
#ax1.set_xlim(xlims)
#ax1.set_ylim(ylims)
ax3.set_ylabel(r"Gain ($E_f / E_L$)")
ax3.set_xlabel(r"Laser energy absorbed")
ax3.legend(loc="lower right")
fig3.savefig(plot_dir + "/" + 'implosion_performance_laser_energy.png', bbox_inches='tight')


## Comparing beamspots with bandwidth

In [ ]:
def fitness_function(dataset, opt_params):
    target_rms = opt_params["fitness_desired_rms"]
    norm_factor = opt_params["fitness_norm_factor"]
    number_of_timesteps = np.shape(dataset["rms"][:,:])[1]

    rms = np.sqrt(np.sum(dataset["rms"][:,:]**2, axis=1) / float(number_of_timesteps))
    avg_flux = np.sqrt(np.sum(dataset["avg_flux"][:,:]**2, axis=1) / float(number_of_timesteps))
    if opt_params["run_plasma_profile"]:
        target_flux = opt_params["fitness_desired_pressure_mbar"]
        indices = np.where(np.array(avg_flux) > opt_params["fitness_limit_broken_pressure_mbar"])[0]
        print("Fitness function detects broken runs: ", indices)
        avg_flux[indices] = float('nan')#0.0
    else:
        target_flux = opt_params["fitness_desired_power_per_steradian"]

    maxi_func = np.exp(-(rms/target_rms) + (avg_flux / target_flux) ** 0.25) * (avg_flux / target_flux)**0.01 * norm_factor
    #maxi_func = np.exp(-(rms/target_rms) + (avg_flux / target_flux) ** 0.25) * (avg_flux / target_flux) * norm_factor
    return maxi_func, indices

def fitness_function_time_dependant(dataset, opt_params):
    target_rms = opt_params["fitness_desired_rms"]
    norm_factor = opt_params["fitness_norm_factor"]
    number_of_timesteps = np.shape(dataset["rms"][:,:])[1]

    rms = dataset["rms"][:,:]
    avg_flux = dataset["avg_flux"][:,:]
    if opt_params["run_plasma_profile"]:
        target_flux = opt_params["fitness_desired_pressure_mbar"]
        indices = np.where(np.array(avg_flux) > opt_params["fitness_limit_broken_pressure_mbar"])[0]
        print("Fitness function detects broken runs: ", indices)
        avg_flux[indices] = float('nan')#0.0
        indices2 = np.where(np.array(avg_flux) < opt_params["fitness_limit_broken_pressure_mbar"] * 0.001)[0]
        print("Fitness function detects broken runs: ", indices2)
    else:
        target_flux = opt_params["fitness_desired_power_per_steradian"]

    maxi_func = np.exp(-(rms/target_rms) + (avg_flux / target_flux) ** 0.25) * (avg_flux / target_flux)**0.01 * norm_factor
    return maxi_func, indices

In [ ]:
diag_dir = "/ccc/scratch/cont002/dam/barlodun/2504a_PDDOptimization/Data/250513g_cpm48_plasma_vary_beamspot_with_bandwidth_partial_fail"
sys_params = tdg.define_system_params(diag_dir)
sys_params["plot_file_type"] = ".png"

aspect_ratio = (5, 4)

In [ ]:
dataset, dataset_params, deck_gen_params, facility_spec = idg.load_data_dicts_from_file(sys_params)
opt_params = uopt.define_optimizer_parameters(diag_dir, 0, 0, dataset_params, facility_spec, sys_params)

initial_dataset = dataset_params["num_examples"] #16#36

fitness_temporal, broken_indices = fitness_function_time_dependant(dataset, opt_params)
fitness_overall, _ = fitness_function(dataset, opt_params)

labels = [None] * dataset_params["num_examples"]
for iconfig in range(dataset_params["num_examples"]):
    labels[iconfig] = r"SG={:.2f} and $R_b/R_t$={:.2f} ".format(deck_gen_params["beamspot_order"][iconfig,0], deck_gen_params["beamspot_major_radius"][iconfig,0]/dataset_params['target_radius'])

In [ ]:
ind_max_fitness = np.argmax(fitness_overall)
print(ind_max_fitness)
print(fitness_overall[ind_max_fitness])
print()

#"""
param1 = deck_gen_params["beamspot_order"][:,0]
label1 = "Super gaussian order"
param2 = deck_gen_params["beamspot_major_radius"][:,0]/dataset_params['target_radius']
label2 = r"Beam to target ratio ($R_b/R_t$)"
semilogy_bool = False
"""
param1 = deck_gen_params["bandwidth_num_spectral_lines"]
label1 = "Number spectral lines"
param2 = deck_gen_params["bandwidth_percentage_width"][:,0]
label2 = r"Bandwidth ($\delta \lambda / \lambda$)"
semilogy_bool = True
#"""

fitness_overall2 = 0.0 * param1
fitness_overall2[:np.shape(fitness_overall)[0]] = fitness_overall


In [ ]:
number_of_timesteps = np.shape(dataset["rms"])[1]

rms_overall = np.sqrt(np.sum(dataset["rms"]**2, axis=1) / float(number_of_timesteps))
avg_flux_overall = np.sqrt(np.sum(dataset["avg_flux"]**2, axis=1) / float(number_of_timesteps))

dataset["rms"][broken_indices] = float("nan")
rms_overall[broken_indices] = float("nan")

num_samples_per_param = int(np.sqrt(initial_dataset)) #int(np.sqrt(dataset_params["num_examples"]))
rms_overall_reshaped = np.reshape(rms_overall[:initial_dataset],(num_samples_per_param,num_samples_per_param))
rms_temporal_reshaped = np.reshape(dataset["rms"][:initial_dataset],(num_samples_per_param,num_samples_per_param,-1))
avg_flux_overall_reshaped = np.reshape(avg_flux_overall[:initial_dataset],(num_samples_per_param,num_samples_per_param))
avg_flux_temporal_reshaped = np.reshape(dataset["avg_flux"][:initial_dataset],(num_samples_per_param,num_samples_per_param,-1))
fitness_overall_reshaped = np.reshape(fitness_overall[:initial_dataset],(num_samples_per_param,num_samples_per_param))
fitness_temporal_reshaped = np.reshape(fitness_temporal[:initial_dataset],(num_samples_per_param,num_samples_per_param,-1))
print(np.shape(fitness_temporal_reshaped), np.shape(avg_flux_temporal_reshaped))
print(np.shape(dataset["input_parameters"]))

#X = np.reshape(dataset["input_parameters"][:,0],(num_samples_per_param,num_samples_per_param))
#Y = np.reshape(dataset["input_parameters"][:,1],(num_samples_per_param,num_samples_per_param))

X = np.reshape(param1[:initial_dataset],(num_samples_per_param,num_samples_per_param))
Y = np.reshape(param2[:initial_dataset],(num_samples_per_param,num_samples_per_param))
print(np.shape(X), np.shape(Y))
#print(rms_overall)

In [ ]:
eval_time_ind=3
fig1, ax1 = plt.subplots(figsize=aspect_ratio, dpi=400)
fig2, ax2 = plt.subplots(figsize=aspect_ratio, dpi=400)
fig3, ax3 = plt.subplots(figsize=aspect_ratio, dpi=400)

cmap = ax1.pcolormesh(X, Y, fitness_overall_reshaped)
fig1.colorbar(cmap, ax=ax1, label="Overall fitness")

cmap = ax2.pcolormesh(X, Y, rms_overall_reshaped * 100., norm=matplotlib.colors.LogNorm(vmax=2))
fig2.colorbar(cmap, ax=ax2, label="Overall rms (%)")

cmap = ax3.pcolormesh(X, Y, avg_flux_temporal_reshaped[:,:,eval_time_ind])
fig3.colorbar(cmap, ax=ax3, label="Peak ablation pressure (Mbars)")

if semilogy_bool:
    ax1.semilogy(param1[ind_max_fitness],param2[ind_max_fitness],"rx")
    ax2.semilogy(param1[ind_max_fitness],param2[ind_max_fitness],"rx")
    ax3.semilogy(param1[ind_max_fitness],param2[ind_max_fitness],"rx")
else:
    ax1.plot(param1[ind_max_fitness],param2[ind_max_fitness],"rx")
    ax2.plot(param1[ind_max_fitness],param2[ind_max_fitness],"rx")
    ax3.plot(param1[ind_max_fitness],param2[ind_max_fitness],"rx")


ax1.set_xlabel(label1)
ax1.set_ylabel(label2)
ax2.set_xlabel(label1)
ax2.set_ylabel(label2)
ax3.set_xlabel(label1)
ax3.set_ylabel(label2)
fig1.savefig(diag_dir+"/scan_results_overall_fitness_grid" + sys_params["plot_file_type"], dpi=300, bbox_inches='tight')
fig2.savefig(diag_dir+"/scan_results_overall_rms_grid" + sys_params["plot_file_type"], dpi=300, bbox_inches='tight')
fig3.savefig(diag_dir+"/scan_results_overall_pabl_grid" + sys_params["plot_file_type"], dpi=300, bbox_inches='tight')

## Compare beam geometry methods

In [ ]:
import numpy as np
import os
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.ticker import MaxNLocator
%matplotlib inline
plt.ion();
import utils_intensity_map as uim
import utils_deck_generation as idg
import netcdf_read_write as nrw
import training_data_generation as tdg
import utils_healpy as uhp
import utils_optimizers as uopt

def fitness_function(dataset, opt_params):
    target_rms = opt_params["fitness_desired_rms"]
    norm_factor = opt_params["fitness_norm_factor"]
    number_of_timesteps = np.shape(dataset["rms"][:,:])[1]

    rms = np.sqrt(np.sum(dataset["rms"][:,:]**2, axis=1) / float(number_of_timesteps))
    avg_flux = np.sqrt(np.sum(dataset["avg_flux"][:,:]**2, axis=1) / float(number_of_timesteps))
    if opt_params["run_plasma_profile"]:
        target_flux = opt_params["fitness_desired_pressure_mbar"]
        indices = np.where(np.array(avg_flux) > opt_params["fitness_limit_broken_pressure_mbar"])[0]
        #print("Fitness function detects broken runs: ", indices)
        avg_flux[indices] = 0.0
    else:
        target_flux = opt_params["fitness_desired_power_per_steradian"]

    maxi_func = np.exp(-(rms/target_rms) + (avg_flux / target_flux) ** 0.33) * (avg_flux / target_flux)**0.01 * norm_factor
    #maxi_func = np.exp(-(rms/target_rms) + (avg_flux / target_flux) ** 0.25) * (avg_flux / target_flux)**0.01 * norm_factor
    return maxi_func

def fitness_function_time_dependant(dataset, opt_params):
    target_rms = opt_params["fitness_desired_rms"]
    norm_factor = opt_params["fitness_norm_factor"]
    number_of_timesteps = np.shape(dataset["rms"][:,:])[1]

    rms = dataset["rms"][:,:]
    avg_flux = dataset["avg_flux"][:,:]
    if opt_params["run_plasma_profile"]:
        target_flux = opt_params["fitness_desired_pressure_mbar"]
        indices1 = np.where(np.array(avg_flux) > opt_params["fitness_limit_broken_pressure_mbar"])[0]
        avg_flux[indices1] = 0.0
        indices = np.where(np.array(avg_flux) < opt_params["fitness_limit_broken_pressure_mbar"] * 0.001)[0]
        print("Fitness function detects broken runs: ", indices1, indices)
    else:
        target_flux = opt_params["fitness_desired_power_per_steradian"]

    maxi_func = np.exp(-(rms/target_rms) + (avg_flux / target_flux) ** 0.25) * (avg_flux / target_flux)**0.01 * norm_factor
    return maxi_func

In [ ]:
list_labels = ["CPM48",
               r"$\Omega$60",
               "ICO80",
               "T72",
               "CPM72",]
list_dirs = [
             "/ccc/scratch/cont002/dam/barlodun/2504a_PDDOptimization/Data/250506i_cpm48_plasma",
             "/ccc/scratch/cont002/dam/barlodun/2504a_PDDOptimization/Data/250429c_lafon351_omega_plasma_more",
             #"/ccc/scratch/cont002/dam/barlodun/2504a_PDDOptimization/Data/250806a_lafon351_omega_plasma_more_optimize_change_fitness",
             "/ccc/scratch/cont002/dam/barlodun/2504a_PDDOptimization/Data/250506a_ico80_plasma",
             "/ccc/scratch/cont002/dam/barlodun/2504a_PDDOptimization/Data/250506d_t11_b72_plasma",
             "/ccc/scratch/cont002/dam/barlodun/2504a_PDDOptimization/Data/250506e_cpm72_plasma",]

In [ ]:
ndirs = len(list_labels)
sys_params = {}
dataset = {}
dataset_params = {}
deck_gen_params = {}
facility_spec = {}
opt_params = {}
fitness_temporal = {}
fitness_overall = {}
label_beam_config = {}

for idir in range(ndirs):
    label = list_labels[idir]
    loc_dir = list_dirs[idir]
    sys_params[label] = {}
    dataset[label] = {}
    dataset_params[label] = {}
    deck_gen_params[label] = {}
    facility_spec[label] = {}
    opt_params[label] = {}
    fitness_temporal[label] = {}
    fitness_overall[label] = {}

    print(label, loc_dir)
    sys_params[label] = tdg.define_system_params(loc_dir)
    dataset[label], dataset_params[label], deck_gen_params[label], facility_spec[label] = idg.load_data_dicts_from_file(sys_params[label])
    opt_params[label] = uopt.define_optimizer_parameters(loc_dir, 0, 0, dataset_params[label], facility_spec[label], sys_params[label])
    #opt_params["fitness_limit_broken_pressure_mbar"] = 1000.0
    print(opt_params[label]["run_plasma_profile"])
    print(dataset[label]["rms"][0])
    print(dataset[label]["avg_flux"][0])

    fitness_temporal[label] = fitness_function_time_dependant(dataset[label], opt_params[label])
    fitness_overall[label] = fitness_function(dataset[label], opt_params[label])

    label_beam_config[label] = [None] * dataset_params[label]["num_examples"]
    for iconfig in range(dataset_params[label]["num_examples"]):
        label_beam_config[label][iconfig] = r"SG={:.2f} and $R_b/R_t$={:.2f} ".format(deck_gen_params[label]["beamspot_order"][iconfig,0], deck_gen_params[label]["beamspot_major_radius"][iconfig,0]/dataset_params[label]['target_radius'])
    label_beam_config[label] = np.array(label_beam_config[label])

In [ ]:
list_colours = [
                "tab:purple",
                "tab:green",
                "tab:blue",
                "tab:pink",
                "tab:orange",
                "tab:cyan",
                "tab:olive",
                "tab:gray",
                "tab:brown",
                "tab:purple",
                "tab:pink",
                "tab:green",
                "tab:orange",
                "tab:blue",
                "tab:cyan",
                "tab:olive",
                "tab:gray",
                "tab:brown",
               ]
list_line_styles = ["solid","dotted","dashed","dashdot","solid","dotted","dashed","dashdot","solid","dotted","dashed","dashdot","solid","dotted","dashed","dashdot"]
list_point_styles = matplotlib.lines.Line2D.markers
print(list_point_styles)

In [ ]:
fig1, ax1 = plt.subplots(figsize=aspect_ratio, dpi=400)

ind_max_fitness = {}
ind_max_fitness_list = {}
ind_max_fitness_temp = {}
ind_min_rms_early_time = {}
rms_overall = {}

for idir in range(ndirs):
    label = list_labels[idir]
    number_of_timesteps = np.shape(dataset[label]["rms"])[1]
    
    ind_max_fitness[label] = np.argmax(fitness_overall[label])
    ind_max_fitness_list[label] = np.array(np.where(fitness_overall[label]>0.5)[0], dtype='int')
    print(label, len(ind_max_fitness_list[label]), np.max(fitness_overall[label]))
    rms_overall[label] = np.sqrt(np.sum(dataset[label]["rms"][:,:]**2, axis=1) / float(number_of_timesteps))
    
    facility_best_bs_rms_overall = rms_overall[label][ind_max_fitness_list[label]]
    
    ind_max_fitness_temp[label] = np.array(np.argmax(fitness_temporal[label][:,-1]), dtype='int')
    facility_best_late_time_bs_rms = dataset[label]["rms"][ind_max_fitness_temp[label],-1]
    facility_best_late_time_bs_rms_early_time = dataset[label]["rms"][ind_max_fitness_temp[label],0]
    
    """
    if label == "CPM72 193nm":
        facility_best_bs_ablation_pressure_late_time = dataset[label]["avg_flux"][ind_max_fitness_list[label],-2]
        facility_best_late_time_bs_ablation_pressure = dataset[label]["avg_flux"][ind_max_fitness_temp[label],-2]
    else:
    """
    facility_best_bs_ablation_pressure_late_time = dataset[label]["avg_flux"][ind_max_fitness_list[label],-1]
    facility_best_late_time_bs_ablation_pressure = dataset[label]["avg_flux"][ind_max_fitness_temp[label],-1]
    
    facility_best_early_time_bs_rms = np.min(dataset[label]["rms"][:,0][np.where(dataset[label]["avg_flux"][:,0]>0.0001)[0]])
    ind_min_rms_early_time[label] = np.where(dataset[label]["rms"][:,0]==facility_best_early_time_bs_rms)[0]
    facility_best_early_time_bs_pabl = dataset[label]["avg_flux"][ind_min_rms_early_time[label],0]

    #print(label, facility_best_bs_rms_overall*100, facility_best_bs_ablation_pressure_late_time)
    ax1.semilogx(facility_best_bs_rms_overall*100, facility_best_bs_ablation_pressure_late_time, "o",
                markersize=5, color=list_colours[idir])#, label=label)
    ax1.semilogx(facility_best_late_time_bs_rms*100, facility_best_late_time_bs_ablation_pressure, "x",
                markersize=5, color=list_colours[idir])
    ax1.semilogx(facility_best_early_time_bs_rms*100, facility_best_early_time_bs_pabl, "^",
                markersize=5, color=list_colours[idir])
    
    sort_key = np.argsort(facility_best_bs_rms_overall)
    line_valx = [facility_best_bs_rms_overall[sort_key[0]]]
    line_valy = [facility_best_bs_ablation_pressure_late_time[sort_key[0]]]
    for ind in range(len(sort_key)):
        if facility_best_bs_ablation_pressure_late_time[sort_key[ind]]>line_valy[-1]:
            line_valx.append(facility_best_bs_rms_overall[sort_key[ind]])
            line_valy.append(facility_best_bs_ablation_pressure_late_time[sort_key[ind]])
    line_valx = np.array(line_valx)
    line_valy = np.array(line_valy)
    ax1.semilogx(line_valx * 100.0, line_valy, color=list_colours[idir], label=label)
    """ 
    ax1.text(rms_overall[label][ind_max_fitness[label]]*100, dataset[label]["avg_flux"][ind_max_fitness[label],-1],
                     label_beam_config[label][ind_max_fitness[label]], fontsize=5, rotation=45)
    ax1.text(facility_best_late_time_bs_rms*100, facility_best_late_time_bs_ablation_pressure,
                     label_beam_config[label][ind_max_fitness_temp[label]], fontsize=5, rotation=45)
    ax1.text(dataset[label]["rms"][ind_min_rms_early_time[label],0]*100, dataset[label]["avg_flux"][ind_min_rms_early_time[label],0],
                     label_beam_config[label][ind_min_rms_early_time[label]][0], fontsize=5, rotation=45)
    
    #"""
    """
    for ind in range(len(ind_max_fitness[label])):
        if (ind % 4 == 0):
            ax1.text(facility_best_bs_rms_overall[label][ind]*100, facility_best_bs_ablation_pressure_late_time[label][ind],
                     label_beam_config[label][ind_max_fitness[label][ind]], fontsize=5)
    """

ax1.legend(loc="upper right")

ax1.set_xlim([0.03,30])
ax1.set_ylim([0,400])
ax1.set_xlabel("RMS ablation pressure (% mean)")
ax1.set_ylabel("Ablation pressure (Mbars)")
fig1.savefig(loc_dir+"/../../"+sys_params[label]["figure_location"]+"/facility_comparison_pabl_and_rms" + plot_file_type, dpi=300, bbox_inches='tight')


## Compare number of beam ports